# Unión de bronze · TFM Energía UCM
Lee los parquets ya generados por `scripts/extract_bronze.py` en `data/bronze/` y arma una vista unida sobre un calendario horario, para consultar como un único dataframe en el EDA.

**Este notebook no se conecta a Postgres.** Si falta el parquet de alguna tabla, avisa y sigue con las demás — no es su trabajo generarlo (eso es `extract_bronze.py`).

**Tres formas de unir, según el `grain` de cada tabla** (definido en `bronze_config.TABLES`):
- `hourly`: join exacto por `ts_utc`.
- `daily`: el valor se coloca solo en la hora `00:00` local de ese día -- no se difunde a las otras 23. Mismo criterio de "no imputar" que el resto.
- `3h`: join **exacto** por `ts_utc`, igual que `hourly` — las horas sin lectura quedan en `NaN`, sin sostener el último valor conocido.

*Asume que corre con working directory `notebooks/` (el default de Jupyter al abrir un notebook desde ahí) para poder importar `bronze_config` como sibling de `scripts/`.*

In [8]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path("..").resolve() / "scripts"))
from bronze_config import BRONZE_DIR, TZ_LOCAL, ANCHOR_TABLE, TABLES, DERIVED_COLUMNS, raw_path, UNIFIED_FILENAME

# Rango de entrenamiento fijado por el equipo -- importado de construir_dataset_maestro.py,
# no reescrito a mano acá, para que no se desincronice si el equipo mueve las fechas.
sys.path.append(str(Path("..").resolve() / "modelos"))
from construir_dataset_maestro import DATASET_START, DATASET_END

## 1 · Configuración de esta corrida
`FECHA_INICIO`/`FECHA_FIN` acotan el calendario al rango de entrenamiento oficial del equipo (`DATASET_START`/`DATASET_END` de `construir_dataset_maestro.py`), no al rango completo de `ANCHOR_TABLE` -- el bronce trae más histórico del que se usa para entrenar (`entsoe_gen_data` arranca en `2019-12-31 23:00 UTC`), y no tiene sentido arrastrar esos días de más a cada EDA. Poner ambas en `None` vuelve al rango completo de `ANCHOR_TABLE`.

In [9]:
FECHA_INICIO = pd.Timestamp(DATASET_START, tz="UTC")
FECHA_FIN = pd.Timestamp(DATASET_END, tz="UTC") + pd.Timedelta(hours=23)  # incluye el día completo, no solo las 00:00

MERGED_OUTPUT_PATH = BRONZE_DIR / UNIFIED_FILENAME

## 2 · Carga de bronze por tabla
Ya vienen normalizadas (`ts_utc` o `date_local` según el `grain` + columnas prefijadas) — este notebook solo las lee.

In [10]:
normalized = {}
for nombre in TABLES:
    path = raw_path(nombre)
    if not path.exists():
        print(f"Aviso: no existe {path} todavía — correr: python scripts/extract_bronze.py {nombre}. Se omite de esta unión.")
        continue
    normalized[nombre] = pd.read_parquet(path)

{nombre: df.shape for nombre, df in normalized.items()}

{'entsoe_gen_data': (58295, 12),
 'esios_gen': (58295, 11),
 'esios_capacity_available': (58337, 7),
 'esios_capacity_installed': (2429, 19),
 'load_inter': (58295, 14),
 'esios_forecast_da': (58343, 5),
 'commodities': (2430, 4),
 'era5_weather_agg': (19279, 10)}

## 3 · Calendario
Anclado a `ANCHOR_TABLE` — no a la unión de todo lo cargado. Así el rango del calendario no cambia según qué tablas resulten estar disponibles en un momento dado; siempre es el mismo eje mientras `ANCHOR_TABLE` no cambie.

In [11]:
def build_calendar(start, end, tz_local=TZ_LOCAL, freq="h") -> pd.DataFrame:
    ts_utc = pd.date_range(start, end, freq=freq, tz="UTC")
    cal = pd.DataFrame({"ts_utc": ts_utc})
    cal["ts_local"] = cal["ts_utc"].dt.tz_convert(tz_local)
    cal["date_utc"] = cal["ts_utc"].dt.date
    cal["date_local"] = cal["ts_local"].dt.date
    cal["hour_utc"] = cal["ts_utc"].dt.hour
    cal["hour_local"] = cal["ts_local"].dt.hour
    return cal

assert ANCHOR_TABLE in normalized, f"Falta el parquet de la tabla ancla ({ANCHOR_TABLE}) — no se puede construir el calendario sin ella."
ancla = normalized[ANCHOR_TABLE]

inicio = FECHA_INICIO or ancla["ts_utc"].min()
fin = FECHA_FIN or ancla["ts_utc"].max()

calendario = build_calendar(inicio, fin)
print(f"Calendario: {inicio} -> {fin}  ({len(calendario)} horas)  anclado en '{ANCHOR_TABLE}'")
calendario.head()

Calendario: 2020-01-01 00:00:00+00:00 -> 2026-08-15 23:00:00+00:00  (58056 horas)  anclado en 'entsoe_gen_data'


,ts_utc,ts_local,date_utc,date_local,hour_utc,hour_local
0,2020-01-01 00:00:00+00:00,2020-01-01 01:00:00+01:00,2020-01-01,2020-01-01,0,1
1,2020-01-01 01:00:00+00:00,2020-01-01 02:00:00+01:00,2020-01-01,2020-01-01,1,2
2,2020-01-01 02:00:00+00:00,2020-01-01 03:00:00+01:00,2020-01-01,2020-01-01,2,3
3,2020-01-01 03:00:00+00:00,2020-01-01 04:00:00+01:00,2020-01-01,2020-01-01,3,4
4,2020-01-01 04:00:00+00:00,2020-01-01 05:00:00+01:00,2020-01-01,2020-01-01,4,5


## 4 · Merge
Cada tabla se une según su `grain`. `hourly` y `3h` se unen **exacto** por `ts_utc` -- las
horas sin lectura quedan en `NaN`, sin rellenar nada. `daily` se coloca **solo en la hora
00 local** de cada día -- no se difunde a las otras 23. Mismo criterio en las tres: el
bronce no imputa, no importa la granularidad de origen.

In [12]:
def unir_exacto(base: pd.DataFrame, df: pd.DataFrame) -> pd.DataFrame:
    """hourly y 3h se unen igual: join exacto por ts_utc. Para 3h, las horas sin
    lectura quedan en NaN a proposito -- no se sostiene el ultimo valor conocido."""
    return base.merge(df, on="ts_utc", how="left")


def unir_daily(base: pd.DataFrame, df: pd.DataFrame) -> pd.DataFrame:
    """El dato es diario -- se coloca SOLO en la hora 00 local de ese dia, sin
    difundir a las otras 23. El resto queda en NaN, mismo criterio que las tablas
    de menor frecuencia (era5, 3h): el bronce no imputa, cualquiera sea el origen."""
    merged = base.merge(df, on="date_local", how="left")
    cols_nuevas = [c for c in df.columns if c != "date_local"]
    merged.loc[merged["hour_local"] != 0, cols_nuevas] = np.nan
    return merged


UNIR_POR_GRAIN = {"hourly": unir_exacto, "3h": unir_exacto, "daily": unir_daily}

bronze_unificado = calendario.copy()
for nombre, df in normalized.items():
    grain = TABLES[nombre].get("grain", "hourly")
    bronze_unificado = UNIR_POR_GRAIN[grain](bronze_unificado, df)

bronze_unificado.shape

(1393183, 80)

## 5 · Columnas derivadas
Se calculan después del merge porque cruzan más de una tabla de origen (ej. FV limpia = ENTSO-E agregada menos termosolar de ESIOS). Definidas en `bronze_config.DERIVED_COLUMNS` -- agregar una nueva ahí, no acá.

In [13]:
for derivada in DERIVED_COLUMNS:
    faltantes = [c for c in derivada["inputs"] if c not in bronze_unificado.columns]
    if faltantes:
        print(f"Aviso: no se puede calcular \'{derivada['name']}\' -- faltan columnas {faltantes}. Se omite.")
        continue
    bronze_unificado[derivada["name"]] = derivada["formula"](bronze_unificado)
    print(f"Calculada: {derivada['name']}")

bronze_unificado.shape

Calculada: calc_solar_fv_mw
Calculada: calc_hydro_dispatch_mw
Calculada: calc_autoconsumo_mw


(1393183, 83)

## 6 · Persistencia

In [14]:
bronze_unificado.to_parquet(MERGED_OUTPUT_PATH, index=False)
print(f"Guardado: {MERGED_OUTPUT_PATH} — {bronze_unificado.shape[0]} filas x {bronze_unificado.shape[1]} columnas")
print(f"Rango real cubierto: {bronze_unificado['ts_utc'].min()} -> {bronze_unificado['ts_utc'].max()}")

Guardado: C:\Users\lhern\git\edev_models\data\bronze\bronze_unificado.parquet — 1393183 filas x 83 columnas
Rango real cubierto: 2020-01-01 00:00:00+00:00 -> 2026-08-15 23:00:00+00:00
